# Notebook 02: Model Benchmark, Calibration, and 38-Country IECV

**Study:** Machine Learning Prediction of Absence of Formal Healthcare Seeking for Acutely Ill Children in Sub-Saharan Africa: A 38-Country IECV Study  
**Target Journal:** *The Lancet Digital Health*  
**DOI:** [10.5281/zenodo.10892345](https://doi.org/10.5281/zenodo.10892345)

This notebook reproduces:
1. **Table 2:** Comparative performance of five ML algorithms in 5-fold stratified cross-validation on pooled cohort (N = 118,910).
2. **Figure 3:** Receiver Operating Characteristic (ROC), Precision-Recall (PR), and Calibration Curves across models.
3. **Table 3 & Table S6:** 38-Country leave-one-country-out Internal-External Cross-Validation (IECV) performance breakdown.
4. **Figure 5:** IECV Forest Plot displaying held-out country AUROCs with parametric Hanley–McNeil 95% confidence intervals.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path("..").resolve()
TABLES_DIR = REPO_ROOT / "results" / "tables"
FIGURES_DIR = REPO_ROOT / "results" / "figures"

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
print("Loaded configuration.")

## 1. Table 2: 5-Fold Stratified Cross-Validation Benchmark

In [ ]:
table2_path = TABLES_DIR / "Table_2.csv"
if not table2_path.exists():
    table2_path = TABLES_DIR / "table2_model_comparison.csv"

df_t2 = pd.read_csv(table2_path)
display(df_t2)

## 2. Table 3: 38-Country Internal-External Cross-Validation (IECV)

In [ ]:
table3_path = TABLES_DIR / "Table_3.csv"
if not table3_path.exists():
    table3_path = TABLES_DIR / "table3_country_iecv_breakdown.csv"

df_t3 = pd.read_csv(table3_path)
display(df_t3)

## 3. Figure 5: 38-Country IECV Forest Plot with Hanley–McNeil 95% CIs

In [ ]:
# Load detailed Table S6 with exact CIs
tables6_path = TABLES_DIR / "Table_S6.csv"
if not tables6_path.exists():
    tables6_path = TABLES_DIR / "table_s6_iecv_diagnostics.csv"

df_s6 = pd.read_csv(tables6_path)

# Extract lower and upper CI bounds
def parse_ci(val_str):
    if '–' in str(val_str):
        parts = val_str.split('(')[1].split(')')[0].split('–')
        return float(parts[0]), float(parts[1])
    elif '-' in str(val_str):
        parts = val_str.split('(')[1].split(')')[0].split('-')
        return float(parts[0]), float(parts[1])
    return np.nan, np.nan

ci_bounds = [parse_ci(s) for s in df_s6['AUROC_95CI']]
df_s6['ci_lo'] = [b[0] for b in ci_bounds]
df_s6['ci_hi'] = [b[1] for b in ci_bounds]

# Sort countries by AUROC ascending for clear forest plot
df_sorted = df_s6.sort_values('AUROC', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 14), dpi=300)
y_pos = np.arange(len(df_sorted))

mean_auc = 0.6967

# Reference lines
ax.axvline(0.50, color='gray', linestyle=':', lw=1.2, label='Null Discrimination (0.50)')
ax.axvline(mean_auc, color='#c5221f', linestyle='--', lw=1.8, label=f'Cross-National Mean AUROC = {mean_auc:.3f} (95% CI: 0.669–0.725)')

# Error bars and points
x_err = [df_sorted['AUROC'] - df_sorted['ci_lo'], df_sorted['ci_hi'] - df_sorted['AUROC']]
ax.errorbar(df_sorted['AUROC'], y_pos, xerr=x_err, fmt='o', color='#2b5c8f',
            ecolor='#6baed6', elinewidth=1.6, capsize=3, markersize=5, label='Held-Out Country Out-of-Sample AUROC')

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{row['Country_Name']} ({row['Country_Code']})" for _, row in df_sorted.iterrows()], fontsize=9)
ax.set_xlabel('Held-Out Country Out-of-Sample AUROC (95% Confidence Interval)', fontsize=11, fontweight='bold')
ax.set_title('Figure 5: 38-Country Leave-One-Country-Out IECV Transportability Forest Plot', fontsize=12, fontweight='bold', pad=15)
ax.set_xlim(0.45, 0.95)
ax.legend(loc='lower right', frameon=True, fontsize=9.5)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure5_iecv_forest_reproduced.png", dpi=300, bbox_inches='tight')
plt.show()